In [13]:
%pip install anthropic python-dotenv
%pip install requests ffmpeg-python
%pip install deepgram-sdk --upgrade
%pip install requests
%pip install anthropic

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [14]:
from dotenv import load_dotenv
load_dotenv("vars.env")

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [15]:
import subprocess
import os
import glob
from openai import OpenAI

client = OpenAI()

with open("golden_set/video_3/it only takes one night… - champ kent (1080p).mp3", "rb") as audio_file:
    transcription = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file,
        prompt="Transcribe the video.",
        response_format="srt",
    )

with open("it only takes one night… - champ kent (1080p).srt", "w") as srt_file:
    srt_file.write(transcription)

print(transcription)

1
00:00:00,000 --> 00:00:04,360
Quentin Tarantino, one of the most acclaimed filmmakers of all time, known for bringing

2
00:00:04,360 --> 00:00:09,040
his bold ideas to life with surreal visuals and mind-bending storytelling.

3
00:00:09,040 --> 00:00:13,400
But he wasn't always a Hollywood icon, once he was just an ordinary guy working at a video

4
00:00:13,400 --> 00:00:14,400
store.

5
00:00:14,480 --> 00:00:17,280
Until one night, he decided to rewrite his story.

6
00:00:17,280 --> 00:00:20,000
And there was like one night, stay up all night long.

7
00:00:20,000 --> 00:00:25,600
And rather than give myself excuses, I would look at everything that I'm in my life or

8
00:00:25,600 --> 00:00:30,080
everything I'm not doing or whatever, and just not give myself any excuses that just

9
00:00:30,080 --> 00:00:31,080
like nail it.

10
00:00:31,080 --> 00:00:35,919
And I would spend like all night laying out everything I'm doing that's wrong.

11
00:00:35,919 --> 00:00:39,680
And th

In [16]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }
    if system:
        params["system"] = system
            
    message = client.messages.create(**params)
        
    # gets handled this way because we need to pick out the text block, not just content[0], since thinking blocks need to be handled
    return "".join(b.text for b in message.content if b.type == "text")

messages = []

while True:
    user_input = input("> ")
    if not user_input.strip():
        break
    add_user_message(messages, user_input)
    answer = chat(messages, system=system_prompt)
    add_assistant_message(messages, answer)
    print("---")
    print(answer)
    print("---")

In [1]:
import re

def _srt_to_text(srt_path):
    with open(srt_path, "r", encoding="utf-8") as f:
        content = f.read()

    lines = []
    for block in re.split(r"\n\s*\n", content.strip()):
        block_lines = block.strip().splitlines()
        # drop index line and timestamp line, keep the rest as spoken text
        for line in block_lines[2:] if len(block_lines) > 2 else block_lines[1:]:
            lines.append(line.strip())

    text = " ".join(lines)
    text = re.sub(r"\s+", " ", text).strip().lower()
    text = re.sub(r"[^\w\s]", "", text)
    return text

def calculate_wer(reference_srt, hypothesis_srt):
    ref_words = _srt_to_text(reference_srt).split()
    hyp_words = _srt_to_text(hypothesis_srt).split()

    n, m = len(ref_words), len(hyp_words)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        dp[i][0] = i
    for j in range(m + 1):
        dp[0][j] = j

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if ref_words[i - 1] == hyp_words[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j], dp[i][j - 1], dp[i - 1][j - 1])

    edits = dp[n][m]
    return edits / n if n > 0 else 0.0

In [19]:
errorscore = calculate_wer("golden_set/video_3/ corrected_transcript.srt", "it only takes one night… - champ kent (1080p).srt")
errorscore

0.007211538461538462